# Notebook 06 — Scenario Failure Analysis

This notebook identifies the **highest-risk ODD conditions** from evaluation results
and produces a prioritised re-test queue — mirroring the Waymo Scenario Operations
methodology of directed testing.

**Workflow**
1. Load evaluation results across all model variants and conditions.
2. Group by scenario dimension (weather, town, time_of_day, traffic_density).
3. Rank conditions by collision rate descending (secondary sort: route completion ascending).
4. Visualise collision rate as a heatmap across weather × traffic_density.
5. Save `results/priority_retest_scenarios.csv` with priority rankings.

**Dependencies**: Run Notebook 04 first to generate `results/eval_results.csv`.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config

cfg = load_config("configs/eval.yaml")

RESULTS_DIR = PROJECT_ROOT / "results"
EVAL_CSV    = RESULTS_DIR / "eval_results.csv"
OUTPUT_CSV  = RESULTS_DIR / "priority_retest_scenarios.csv"

print(f"Eval towns  : {cfg['eval_towns']}")
print(f"Eval weathers: {cfg['eval_weathers']}")
print(f"Results dir : {RESULTS_DIR}")

In [ ]:
# ── Load evaluation results ─────────────────────────────────────────────────
import pandas as pd

try:
    df = pd.read_csv(EVAL_CSV)
    print(f"Loaded {len(df)} episode records from {EVAL_CSV}")
    print(df.head())
except FileNotFoundError:
    print(
        f"[ERROR] {EVAL_CSV} not found.\n"
        "Run Notebook 04 (evaluation) first to generate this file.\n"
        "Expected columns: model, weather, town, time_of_day, traffic_density, "
        "collision_rate, route_completion, lane_keep, avg_speed_kmh"
    )
    df = pd.DataFrame()  # allow remaining cells to be inspected without crashing

In [ ]:
# ── Rank conditions by collision rate ───────────────────────────────────────
if df.empty:
    print("No data — skipping analysis.")
else:
    # Aggregate over all model variants and towns; average collision_rate
    # and route_completion per (weather, town, time_of_day, traffic_density)
    GROUP_COLS = ["weather", "town", "time_of_day", "traffic_density"]

    condition_stats = (
        df.groupby(GROUP_COLS)
        .agg(
            collision_rate=("collision_rate", "mean"),
            route_completion=("route_completion", "mean"),
            n_episodes=("collision_rate", "count"),
        )
        .reset_index()
        .sort_values(
            by=["collision_rate", "route_completion"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    condition_stats["priority_rank"] = condition_stats.index + 1
    # Assign a stable condition_id for downstream referencing
    condition_stats.insert(
        0, "condition_id",
        condition_stats.apply(
            lambda r: f"{r['weather']}_{r['town']}_{r['time_of_day']}_{r['traffic_density']}",
            axis=1,
        ),
    )

    print(f"Total unique conditions: {len(condition_stats)}")
    print("\nTop 10 highest-risk conditions:")
    print(
        condition_stats[
            ["priority_rank", "condition_id", "collision_rate", "route_completion", "n_episodes"]
        ].head(10).to_string(index=False)
    )

In [ ]:
# ── Heatmap: collision rate across weather × traffic_density ────────────────
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty:
    pivot = (
        df.groupby(["weather", "traffic_density"])["collision_rate"]
        .mean()
        .unstack("traffic_density")
    )
    # Reorder traffic density columns if present
    ordered_traffic = [c for c in ["low", "medium", "high"] if c in pivot.columns]
    pivot = pivot[ordered_traffic] if ordered_traffic else pivot

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        linewidths=0.5,
        ax=ax,
        cbar_kws={"label": "Mean collision rate"},
    )
    ax.set_title("Collision Rate by Weather × Traffic Density\n(averaged over all models and towns)", pad=12)
    ax.set_xlabel("Traffic Density")
    ax.set_ylabel("Weather Preset")
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "collision_heatmap_weather_traffic.png", dpi=150)
    plt.show()
    print("Heatmap saved to results/collision_heatmap_weather_traffic.png")

In [ ]:
# ── Save priority_retest_scenarios.csv ──────────────────────────────────────
if not df.empty:
    output_cols = [
        "condition_id",
        "weather",
        "town",
        "time_of_day",
        "traffic_density",
        "collision_rate",
        "route_completion",
        "priority_rank",
    ]
    condition_stats[output_cols].to_csv(OUTPUT_CSV, index=False)
    print(f"Priority re-test queue saved: {OUTPUT_CSV} ({len(condition_stats)} conditions)")
else:
    print("Skipped — no data loaded.")

## Summary

The priority re-test queue surfaces the ODD conditions most likely to produce
collisions or incomplete routes.  Based on the causal analysis (Notebook 05),
the expected top-5 highest-risk combinations are:

| Rank | Weather | Time of Day | Traffic | Driver |
|------|---------|-------------|---------|--------|
| 1 | HardRainNoon | Night | High | Hurry |
| 2 | HardRainNoon | Sunset | High | Hurry |
| 3 | WetNoon | Night | High | Hurry |
| 4 | HardRainNoon | Night | Medium | Hurry |
| 5 | ClearNight | — | High | Standard |

*(Exact ranking depends on completed evaluation run results.)*

**Recommended next steps**
1. Increase episode count from 10 to 50 for the top-10 ranked conditions to
   tighten confidence intervals on collision rate.
2. Add adversarial weather variants (fog, snow) within the highest-risk
   weather–traffic cells to extend ODD coverage.
3. Use `priority_retest_scenarios.csv` as the regression gate input: after any
   model checkpoint update, re-run evaluation on top-20 conditions and flag any
   increase in collision rate beyond one bootstrapped-CI width as a regression.
4. Investigate whether PPO hurry style can be re-tuned to reduce collision rate
   in high-density conditions without sacrificing route completion gains.